# TensorFlow lab: MNIST classification with `tf.data`

In this exercise you'll build an image classification pipeline end to end: turn a folder of PNG files into a `tf.data.Dataset`, train a small CNN with `tf.keras`, and run inference on a single raw image.

**How to work through this notebook**
- Every task cell is marked `# TODO`. Replace the placeholder with your own code.
- Several tasks end with an `assert` — it will raise with an explanatory message if the task isn't done yet. Don't remove or edit the asserts; they exist to catch mistakes early, before a confusing error shows up three cells later.
- Run cells top to bottom. Later tasks depend on variables defined in earlier ones (`mnist_df`, `make_dataset`, `model`, ...).
- This notebook is self-contained: it only assumes a `data/mnist/<split>/<class>/<file>.png` folder layout (adjust `MNIST_DIR` below if yours differs).

## 1. Imports

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

## 2. Paths

`MNIST_DIR` should point at a folder containing `train/<class>/*.png` and `test/<class>/*.png`. `CHECKPOINT_PATH` is where the training callback (task 8) will save the best model weights.

In [ ]:
MNIST_DIR = Path('../mnist')  # adjust if your MNIST folder lives elsewhere
CHECKPOINT_PATH = Path('mnist_best.weights.h5')

assert MNIST_DIR.exists(), f'No such directory {MNIST_DIR}'

## 3. Task: build a manifest DataFrame

Walk `MNIST_DIR` and build a `pandas.DataFrame` describing every image, with one row per file and three columns:

- `path` — the image's path (e.g. relative to `MNIST_DIR`), as a string
- `class` — the digit label (`0`-`9`), as an int
- `split` — `'train'` or `'test'`

The label and split can both be read straight off the folder structure: `<split>/<class>/<file>.png`.

Expected shape, e.g.:

| path | class | split |
|---|---|---|
| train/0/1.png | 0 | train |
| train/7/842.png | 7 | train |
| test/3/91.png | 3 | test |

Saving it to a CSV (e.g. with `mnist_df.to_csv(...)`) is optional but is handy if you want to reload the manifest later without re-scanning the filesystem.

In [ ]:
# TODO: build the manifest dataframe described above
mnist_df = None

assert mnist_df is not None, 'Task 3 is not done yet: build mnist_df before continuing.'

## 4. Task: visualize random samples

Sample a handful of rows from `mnist_df` and plot them in a grid with their labels as titles, as a sanity check on the manifest.

Hints: `mnist_df.sample(...)`, `cv2.imread(path, cv2.IMREAD_GRAYSCALE)` to load an image, `plt.subplots(...)` + `ax.imshow(...)` to build the grid.

In [ ]:
# TODO: visualize a grid of random samples from mnist_df, with their labels as titles

## 5. Task: build a `tf.data.Dataset`

[`tf.data.Dataset`](https://www.tensorflow.org/guide/data) ([API reference](https://www.tensorflow.org/api_docs/python/tf/data/Dataset)) represents a pipeline of elements plus a chain of transformations (map, filter, batch, ...) applied to them lazily, one element at a time, as the model asks for data. Building the pipeline out of native TensorFlow ops (instead of e.g. loading everything into a big NumPy array upfront) means it can stream from disk, run in parallel, and overlap loading with training.

The pipeline you'll build goes:

1. **`path_ds` / `label_ds`** — two `tf.data.Dataset`s built straight from `mnist_df`'s columns with `tf.data.Dataset.from_tensor_slices`: one of file path strings, one of integer labels.
2. **`zip`** — `tf.data.Dataset.zip((path_ds, label_ds))` pairs them up, so each element becomes a `(path, label)` tuple.
3. **`shuffle`** *(optional)* — `.shuffle(buffer_size)` randomizes the order elements come out in, so training doesn't see whole runs of the same class back to back just because of how the files happen to be sorted on disk. Skip it for validation/test data, where order doesn't matter.
4. **load file** — `tf.io.read_file(path)` reads the PNG's raw bytes off disk.
5. **parse image** — `tf.io.decode_png(bytes, channels=1)` decodes those bytes into an actual image tensor.
6. **normalize** — rescale pixel values from the `[0, 255]` integer range to `[0, 1]` floats, e.g. with `tf.image.convert_image_dtype(image, tf.float32)`. Models train more reliably on small float inputs than on raw byte values.
7. **`batch`** *(optional)* — `.batch(batch_size)` groups consecutive elements into mini-batches. Leaving it `None` yields one example at a time, which is what we want below for inspection and visualization.
8. **`prefetch`** — `.prefetch(tf.data.AUTOTUNE)` lets TensorFlow prepare the next element(s) on CPU while the current one is still being consumed, overlapping I/O with computation instead of doing them one after another.

Fill in `make_dataset` below, then use it to build an unbatched dataset for each split.

In [ ]:
def make_dataset(df, shuffle=False, batch_size=None):
    # TODO: implement the pipeline described above and return the resulting dataset
    pass

In [ ]:
train_dataset_no_batch = make_dataset(mnist_df[mnist_df.split == 'train'], shuffle=True, batch_size=None)
test_dataset_no_batch = make_dataset(mnist_df[mnist_df.split == 'test'], shuffle=False, batch_size=None)

assert train_dataset_no_batch is not None, 'Task 5 is not done yet: implement make_dataset before continuing.'
assert test_dataset_no_batch is not None, 'Task 5 is not done yet: implement make_dataset before continuing.'

print(train_dataset_no_batch)

## 6. Task: visualize images from the dataset

We deliberately built `train_dataset_no_batch` *unbatched* — every element is a single `(image, label)` pair rather than a batch of them, which is exactly what you want when you're plotting one image per grid cell. Batching now would just add an extra leading dimension you'd have to index away before calling `imshow`.

Pull a handful of elements out with `train_dataset_no_batch.take(n)` and plot them with `matplotlib`, the same way you did in task 4 — except this time the images are coming through the `tf.data` pipeline you just built, not straight from `cv2.imread`. This is a good way to confirm loading, decoding, and normalization are all correct.

In [ ]:
# TODO: visualize a grid of images pulled from train_dataset_no_batch, with their labels as titles

## 7. Task: define the model

Build a [`tf.keras.Sequential`](https://www.tensorflow.org/guide/keras/sequential_model) model for 10-class digit classification.

`Sequential` is the simplest way to build a Keras model: a plain stack of layers, each with exactly one input and one output, feeding directly into the next. You build it by passing a list of layers in order — it's the right tool whenever your model is a single linear chain (as ours is here); if you ever need multiple inputs/outputs, shared layers, or branches/skip connections, that's what the Functional API is for instead.

A reasonable architecture for MNIST: an `Input` layer for the `(28, 28, 1)` images, one or two `Conv2D` + `MaxPooling2D` blocks to extract features, a `Flatten`, then a `Dense` head ending in a 10-unit `softmax` output layer.

In [ ]:
# TODO: build your tf.keras.Sequential model here
model = None

assert model is not None, 'Task 7 is not done yet: define your Sequential model before continuing.'

model.summary()

## 8. Task: compile the model

Compile with the standard pattern:

- `optimizer='adam'`
- a loss suited to integer (not one-hot) class labels
- `metrics=['accuracy']`

Then create a [`tf.keras.callbacks.ModelCheckpoint`](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ModelCheckpoint) that watches validation accuracy and saves only the best epoch's weights to `CHECKPOINT_PATH`.

In [ ]:
# TODO: model.compile(...) with the optimizer/loss/metrics described above

# TODO: build a ModelCheckpoint callback that saves the best weights to CHECKPOINT_PATH
checkpoint_callback = None

## 9. Task: train the model

Build batched train/val datasets with `make_dataset` (this time passing `batch_size=BATCH_SIZE`), then train with [`model.fit`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#fit), passing `validation_data=val_ds`, `epochs=EPOCHS`, and `callbacks=[checkpoint_callback]`.

In [ ]:
BATCH_SIZE = 32
EPOCHS = 5

# TODO: build batched train/val datasets using make_dataset
train_ds = None
val_ds = None

# TODO: train the model with model.fit, passing validation_data and the checkpoint callback
history = None

## 10. Task: visualize training metrics

`history.history` is a dict of per-epoch metric lists (`'loss'`, `'val_loss'`, `'accuracy'`, `'val_accuracy'`). Turn it into a `pandas.DataFrame` and plot loss and accuracy (train vs. validation) to check for over/underfitting.

In [ ]:
# TODO: plot loss vs. val_loss, and accuracy vs. val_accuracy, from history.history

## 11. Task: inference on a custom sample

Run the trained model on a single raw image file, end to end:

1. **Load** the best checkpoint (`model.load_weights(CHECKPOINT_PATH)`), then read one PNG file straight from `MNIST_DIR` with `cv2.imread`.
2. **Normalize** it the same way the training pipeline did (scale to `[0, 1]` floats).
3. **To tensor** — reshape it to a batch of one, `(1, 28, 28, 1)`, matching what the model expects.
4. **Predict** with `model.predict(...)`.
5. **Decode** the prediction: `np.argmax` over the 10 output probabilities gives the predicted digit.

Display the image with the true label (from the filename/folder) and the predicted label side by side.

In [ ]:
# TODO: load the best checkpoint, load + normalize one raw image, predict, and display true vs. predicted labels